In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import pickle
import deepRD.tools.trajectoryTools as trajectoryTools
import deepRD.tools.analysisTools as analysisTools
import csv
import math
from deepRD.noiseSampler import noiseSampler
from torchvision.transforms import ToTensor
from torch import nn
from torch.utils.data import DataLoader
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from scipy.stats import gaussian_kde, wasserstein_distance_nd

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
"""
    Comparing trajectories between the benchmark simulation for quarter timestep (0.0125) and the 
    reduced coarse-grained trajectory with timestep x2 or x4 (0.025 or 0.05)
""";

In [ ]:
def apply_periodic(q, boxsize):
    """
    Apply periodic boundary conditions to positions q.

    q       : [..., 3] (torch.Tensor)
    boxsize : float or array-like of length 3
              box spans [-L/2, L/2] in each dimension
    """
    if not torch.is_tensor(q):
        q = torch.as_tensor(q, dtype=torch.float32)

    if not torch.is_tensor(boxsize):
        boxsize = torch.as_tensor(boxsize, dtype=q.dtype, device=q.device)

    if boxsize.ndim == 0:
        boxsize = boxsize.expand(3)

    # Map to [0, L), then shift back to [-L/2, L/2)
    q_shifted = q + boxsize / 2.0
    q_wrapped = torch.remainder(q_shifted, boxsize)
    q_periodic = q_wrapped - boxsize / 2.0
    return q_periodic


def bistable_force(q, minimaDist, kconstants, scale=1.0):
    """
    Vectorised bistable force for positions q.

    q         : [..., 3]  (any leading batch dims, torch.Tensor)
    minimaDist: float
    kconstants: (3,) array-like or tensor [kx, ky, kz]
    scale     : float

    Returns:
        force: [..., 3]
    """
    if not torch.is_tensor(q):
        q = torch.as_tensor(q, dtype=torch.float32)

    kx, ky, kz = kconstants
    x = q[..., 0]
    y = q[..., 1]
    z = q[..., 2]

    force = torch.zeros_like(q)
    force[..., 0] = - kx * 4 * x * (x**2 - minimaDist**2) / (minimaDist**4)
    force[..., 1] = - ky * 2 * y
    force[..., 2] = - kz * 2 * z

    return scale * force


def aboba_deterministic_step(q_n, v_n, dt_eff, Gamma, mass,
                             minimaDist, kconstants, boxsize, scale=1.0):
    """
    Deterministic ABOBA step (no noise) for one time step dt_eff.

    q_n, v_n : [..., 3]
    dt_eff   : float (effective dt = step * dt)
    Gamma    : friction scalar
    mass     : mass scalar
    """
    # A: first half-step in position
    q_half = q_n + v_n * (dt_eff / 2.0)
    #q_half = apply_periodic(q_half, boxsize)

    # Force at x^{n+1/2}
    F_half = bistable_force(q_half, minimaDist, kconstants, scale=scale)

    # BOB step without noise:
    # expterm = exp(-Gamma * dt / mass)
    # frictionForceTerm = v_n * expterm + (1 + expterm) * F * dt/(2m)
    expterm = torch.exp(torch.tensor(-Gamma * dt_eff / mass,
                                     dtype=q_n.dtype, device=q_n.device))

    v_det = v_n * expterm + (1.0 + expterm) * F_half * (dt_eff / (2.0 * mass))

    # A: second half-step in position
    q_next = q_half + v_det * (dt_eff / 2.0)
    q_next = apply_periodic(q_next, boxsize)

    return q_next, v_det

def compute_r_cg_from_fine(q, v, step, parameters,
                           boxsize=5.0, scale=1.0):
    """
    Compute coarse-grained interaction noise r_cg from fine benchmark trajectories,
    using the same ABOBA scheme as langevinNoiseSampler but *without* noise.

    q, v : [n_traj, T, 3]  (fine dt)
    dt   : fine timestep
    step : integer k (dt_eff = k * dt)

    Gamma, mass       : friction and mass
    minimaDist        : bistable parameter
    kconstants        : (3,) for bistable (kx, ky, kz)
    boxsize           : scalar or (3,) – periodic box, interval [-L/2, L/2]
    scale             : scale factor for the potential

    Returns:
        q_n  : [n_traj, T-step, 3] starting positions (at time n)
        v_n  : [n_traj, T-step, 3] starting velocities
        r_cg : [n_traj, T-step, 3] coarse noise increments s.t.

            v_{n+step}^bench = v_det(q_n, v_n; dt_eff) + r_cg(n)
    """
    assert q.shape == v.shape
    assert q.ndim == 3 and q.shape[-1] == 3, "q, v must be [n_traj, T, 3]"
    assert step >= 1
    
    dt = parameters['dt']
    Gamma = parameters['Gamma']
    mass = parameters['mass']

    n_traj, T, _ = q.shape
    T_cg = T - step
    if T_cg <= 0:
        raise ValueError(f"Not enough timesteps T={T} for step={step}.")

    dt_eff = step * dt

    # states at time n
    q_n    = q[:, :T_cg, :]    # [n_traj, T-step, 3]
    v_n    = v[:, :T_cg, :]    # [n_traj, T-step, 3]

    # benchmark velocities at time n+step
    v_next = v[:, step:, :]    # [n_traj, T-step, 3]

    # deterministic ABOBA step with dt_eff
    q_det, v_det = aboba_deterministic_step(
        q_n, v_n,
        dt_eff=dt_eff,
        Gamma=Gamma,
        mass=mass,
        minimaDist=minimaDist,
        kconstants=kconstants,
        boxsize=boxsize,
        scale=scale,
    )

    # coarse interaction noise
    r_cg = v_next - v_det # [n_traj, T-step, 3]
    
    q_cg = q[:, step:, :] # aligning trajectories
    v_cg = v[:, step:, :]

    return q_cg, v_cg, r_cg

In [ ]:
# System type: 'bistable', 'dimer'
systemType = 'bistable'

# datapoint = [time (1), qi (3), vi (3), ? (1), ri(3)] -- 11 dim for benchmark
# for dimer, alternating between particle 1 and particle 2.

#for generated trajs, 
# datapoint = [time (1), qi (3), vi (3), ri(3)] - 10 dim

# Datasets directory
localDirectoryBase = "/group/ag_cmb/scratch/maojrs/stochasticClosure/" + systemType + "/boxsize5/"

# loading multiple simulation folders for comparisons, and corresponding labels

# Conditioning variables: piri, piririm, pipimri, etc. - for dimer, piridqi
datasetFolders = ["benchmark_quarter_ReducedGen2_piri/", "benchmark_quarter_ReducedGen2_piririm/"]#, "benchmarkReducedGen4_piririm/"]
datasetLabels = ["piri_4", "piririm_4"]#, "piririm_2, Tr=0.6"]
nModels = len(datasetFolders)
stride = 4 # stride in the trajectory integration

nTrajs = 100 # no. of trajectories to load per data folder

nTimesteps = 800000//stride # length of coarse-grained trajectory
datapointDim = 10 # dimensionality of datapoints.
numFilesBench = 2500
numFiles = 100 # total no. of files in data folder

#mean_d = torch.tensor([ 4.9998e+02, -1.6760e-02,  3.3266e-03, -1.7140e-03, -1.2232e-05, 
#                        -7.2127e-05,  6.9702e-05, 2.0080e-07,  6.3679e-06, -2.9733e-06])
#std_d = torch.tensor([1.4434e+02, 1.3641e+00, 7.3784e-01, 7.3927e-01, 1.4253e-01, 1.4252e-01, 
#                    1.4256e-01, 1.6200e-02, 1.6212e-02, 1.6207e-02])

# Initialising tensor to store trajectories
dataset = torch.empty((nModels, nTrajs, nTimesteps, datapointDim))
    
# Loading reduced models data

truncate = False # if True, will select random fragment of trajectory

for i, datasetFolder in enumerate(datasetFolders):
    
    filePath = localDirectoryBase + datasetFolder + "simMoriZwanzigReduced_"    
    
    fileIDS = np.sort(np.random.choice(numFiles, nTrajs, replace=False))
    if i==1:
        print(fileIDS)
    
    for j, fnum in enumerate(fileIDS):
        try:
            ds = torch.Tensor(trajectoryTools.loadTrajectory(filePath, fnum))
        except FileNotFoundError:
            print(f'File {fnum} not available')
            continue
            
        dataset[i, j] = ds
        print(f'Model {i+1}: File no. {fnum} loaded', end='\r')

#dataset_norm = (dataset[0] - mean_d)/std_d
dataset.shape # nModels, nTrajs, nTimesteps, datapointDims

In [ ]:
# Loading Benchmark data
nTimestepsBench = 40000 # length of benchmark trajectory

benchDataset = torch.empty((nTrajs, nTimestepsBench, datapointDim))

fileIDS = np.sort(np.random.choice(numFilesBench, nTrajs, replace=False))

benchFileDirectory = localDirectoryBase + "benchmark_quarter_dt/simMoriZwanzig_"

for j, fnum in enumerate(fileIDS):
    try:
        ds = torch.Tensor(trajectoryTools.loadTrajectory(benchFileDirectory, fnum))
    except FileNotFoundError:
        print(f'File {fnum} not available')
        continue

    # cutting out meaningless variable
    if ds.shape[1]==11:
        ds = torch.cat((ds[:, :7], ds[:, -3:]), dim=1)
    
    benchDataset[j] = ds
    
benchDataset.shape

In [ ]:
# Load parameters from parameters file
parameterDictionary = analysisTools.readParameters(localDirectoryBase+"benchmark_quarter_dt/" + "parameters")

# Parameters for external potential (will only acts on distinguished particles (type 1))
minimaDist = 1.5
kconstants = np.array([1.0, 1.0, 1.0])
scalefactor = 1

b_qT, b_vT, b_rT = compute_r_cg_from_fine(benchDataset[..., 1:4], benchDataset[..., 4:7], step=stride, parameters=parameterDictionary)

b_qT = b_qT[:, ::stride, :]
b_vT = b_vT[:, ::stride, :]
b_rT = b_rT[:, ::stride, :]

print(parameterDictionary)

In [ ]:
b_timesteps = benchDataset[:, 2::stride, 0]
#b_qT = benchDataset[:, 2::stride, 1:4]
#b_rT = benchDataset[:, 2::stride, -3:]
#b_vT = benchDataset[:, 2::stride, 4:7]

b_rNxtT = torch.roll(b_rT, -1, 1)


timesteps = dataset[:, :, :, 0]
qT = dataset[:, :, :, 1:4]
rT = dataset[:, :, :, -3:]
rNxtT = torch.roll(rT, -1, 2)
vT = dataset[:, :, :, 4:7]

b_timesteps.shape, b_qT.shape, b_rT.shape, b_rNxtT.shape, b_vT.shape, timesteps.shape, qT.shape, rT.shape, rNxtT.shape, vT.shape

In [ ]:
print('Velocity')

print(f'Mean Bench: \t', torch.mean(b_vT, dim=(0,1)))
print('Std Bench: \t', torch.std(b_vT, dim=(0,1)), '\n')


for i in range(nModels):
    print(f'Mean Reduced {i+1}:\t', torch.mean(vT[i], dim=(0,1)))
    print(f'Std Reduced {i+1}:\t', torch.std(vT[i], dim=(0,1)), '\n')

    
print('\n Auxiliary Var')
print('Mean Bench: \t', torch.mean(b_rT, dim=(0,1)))
print('Std Bench: \t', torch.std(b_rT, dim=(0,1)), '\n')

for i in range(nModels):
    print(f'Mean Reduced {i+1}:\t', torch.mean(rT[i], dim=(0,1)))
    print(f'Std Reduced {i+1}:\t', torch.std(rT[i], dim=(0,1)), '\n')

In [ ]:
colors = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]

def plot_pos_vel_distributions(b_qT, b_vT, qT, vT, model_labels=None, n_samples=50000):
    """
    Plots position and velocity marginal distributions for benchmark and reduced models.

    Args:
        b_qT : tensor [n_traj, n_steps, 3] -- benchmark positions
        b_vT : tensor [n_traj, n_steps, 3] -- benchmark velocities
        qT   : tensor [n_models, n_traj, n_steps, 3]
        vT   : tensor [n_models, n_traj, n_steps, 3]
        model_labels : list of str of length n_models (optional)
        n_samples : int -- number of random samples from each source
    """

    # -------------------------------------------------------
    # 1. Prepare labels
    # -------------------------------------------------------
    n_models = qT.shape[0]
    if model_labels is None:
        model_labels = [f"Model {i}" for i in range(n_models)]

    # -------------------------------------------------------
    # 2. Random subsampling (for speed)
    # -------------------------------------------------------
    def sample_tensor(x, maxN):
        flat = x.reshape(-1, x.shape[-1])
        N = flat.shape[0]
        idx = np.random.choice(N, size=min(N, maxN), replace=False)
        return flat[idx].cpu().numpy()

    b_q_s = sample_tensor(b_qT, n_samples)   # [N, 3]
    b_v_s = sample_tensor(b_vT, n_samples)

    q_s = [sample_tensor(qT[i], n_samples) for i in range(n_models)]
    v_s = [sample_tensor(vT[i], n_samples) for i in range(n_models)]

    # -------------------------------------------------------
    # 3. Plotting
    # -------------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    dims = ["x", "y", "z"]

    # --------------------------
    # Position distributions
    # --------------------------
    for d in range(3):
        ax = axes[0, d]

        # Benchmark KDE
        kde_b = gaussian_kde(b_q_s[:, d])
        xs = np.linspace(b_q_s[:, d].min(), b_q_s[:, d].max(), 400)
        ax.plot(xs, kde_b(xs), color="black", lw=2, label="Benchmark")

        # Reduced models
        for m in range(n_models):
            kde_m = gaussian_kde(q_s[m][:, d])
            ax.plot(xs, kde_m(xs), ls="--", lw=1.5,
                    color=colors[m % len(colors)],
                    label=model_labels[m] if d == 0 else None)

        ax.set_title(f"Position distribution: {dims[d]}")
        ax.set_xlabel(f"q_{dims[d]}")
        ax.set_ylabel("Density")

    # --------------------------
    # Velocity distributions
    # --------------------------
    for d in range(3):
        ax = axes[1, d]

        # Benchmark KDE
        kde_b = gaussian_kde(b_v_s[:, d])
        xs = np.linspace(b_v_s[:, d].min(), b_v_s[:, d].max(), 400)
        ax.plot(xs, kde_b(xs), color="black", lw=2, label="Benchmark")

        # Reduced models
        for m in range(n_models):
            kde_m = gaussian_kde(v_s[m][:, d])
            ax.plot(xs, kde_m(xs), ls="--", lw=1.5,
                    color=colors[m % len(colors)],
                    label=model_labels[m] if d == 0 else None)

        ax.set_title(f"Velocity distribution: {dims[d]}")
        ax.set_xlabel(f"v_{dims[d]}")
        ax.set_ylabel("Density")

    # --------------------------
    # Final formatting
    # --------------------------
    handles, labels = axes[0,0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
def correlation_fft(a, b, trunc):
    """Calculates correlation via FFT."""
    len_a = len(a)
    a = np.concatenate([a, np.zeros(len_a)])
    b = np.concatenate([b, np.zeros(len_a)])
    a_fft = np.fft.fft(a)
    b_fft = np.fft.fft(b)
    corr = np.fft.ifft(a_fft * np.conj(b_fft))
    corr = corr[:trunc].real
    corr /= np.linspace(len_a, len_a - trunc + 1, trunc)
    return corr

In [ ]:
# CALCULATING AUTOCORRELATION FUNCTIONS.
#print(sind, eind)
lag_fine = 2000
lagtimesteps = lag_fine//stride
mTrajs = 20
t_lags = np.arange(lagtimesteps)*0.05*stride

# ACF by FFT for 1-D traj, then summing up over vector dimensions and all trajs 
ACF_FFT = np.zeros((1+nModels, 2, lagtimesteps))
    
for trajInd in np.random.choice(nTrajs, mTrajs):
    
    #print('Benchmark')
    # position
    ACF_FFT[0, 0] += np.sum([correlation_fft(b_qT[trajInd, :, i], b_qT[trajInd, :, i], lagtimesteps) for i in range(b_qT.shape[2])], axis=0)
    # velocity
    ACF_FFT[0, 1] += np.sum([correlation_fft(b_vT[trajInd, :, i], b_vT[trajInd, :, i], lagtimesteps) for i in range(b_vT.shape[2])], axis=0)
    
    for i in range(nModels):
        #print('Reduced')
        ACF_FFT[i+1, 0] += np.sum([correlation_fft(qT[i, trajInd, :, d], qT[i, trajInd, :, d], lagtimesteps) for d in range(qT.shape[3])], axis=0)
        ACF_FFT[i+1, 1] += np.sum([correlation_fft(vT[i, trajInd, :, d], vT[i, trajInd, :, d], lagtimesteps) for d in range(vT.shape[3])], axis=0)
    
    #ACF_FFT[0] += np.sum([correlation_fft(b_vTraj[:, i], b_vTraj[:, i], lagtimesteps) for i in range(b_vTraj.shape[1])], axis=0)
    #ACF_FFT[1] += np.sum([correlation_fft(vTraj[:, i], vTraj[:, i], lagtimesteps) for i in range(vTraj.shape[1])], axis=0)
    
ACF_FFT[0, 0] /= ACF_FFT[0, 0, 0]
ACF_FFT[0, 1] /= ACF_FFT[0, 1, 0]

for i in range(nModels):
        ACF_FFT[i+1, 0] /= ACF_FFT[i+1, 0, 0]
        ACF_FFT[i+1, 1] /= ACF_FFT[i+1, 1, 0]

In [ ]:
def plot_acf_results(ACF_FFT, t_lags, reducedLabels):
    """
    Plot distance (position) ACF and velocity ACF for benchmark + reduced models.

    ACF_FFT: array of shape [1 + nModels, 2, lag]
        [:, 0, :] → position ACF
        [:, 1, :] → velocity ACF
    t_lags: time axis in physical units
    labels: list of labels, length 1 + nModels
    """
    nModels = ACF_FFT.shape[0] - 1
    assert len(reducedLabels) == nModels
    
    labels = ["Benchmark"]
    for i in range(nModels):
        labels.append(reducedLabels[i])
        
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --------------------------
    # Position ACF (left panel)
    # --------------------------
    ax = axes[0]
    for i in range(nModels + 1):
        ax.plot(
            t_lags, 
            ACF_FFT[i, 0], 
            label=labels[i], 
            lw=2 if i == 0 else 1.5,
            color="black" if i==0 else colors[i-1],
            linestyle='-' if i == 0 else '--'
        )
    ax.set_title("Position ACF", fontsize=14)
    ax.set_xlabel("time (ns)", fontsize=12)
    ax.set_ylabel("ACF", fontsize=12)
    ax.legend()
    ax.grid(alpha=0.3)

    # --------------------------
    # Velocity ACF (right panel)
    # --------------------------
    ax = axes[1]
    for i in range(nModels + 1):
        ax.plot(
            t_lags, 
            ACF_FFT[i, 1], 
            label=labels[i],
            lw=2 if i == 0 else 1.5,
            color="black" if i==0 else colors[i-1],
            linestyle='-' if i == 0 else '--'
        )
    ax.set_title("Velocity ACF", fontsize=14)
    ax.set_xlabel("time (ns)", fontsize=12)
    ax.set_ylabel("ACF", fontsize=12)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_pos_vel_distributions(
    b_qT, b_vT,
    qT, vT,
    model_labels=datasetLabels
)

In [ ]:
plot_acf_results(ACF_FFT, t_lags, datasetLabels)

In [ ]:
#Calculating the Mean Squared Displacement of the particle.
b_diffsq = torch.diff(torch.norm(b_qT, dim=-1), dim=-1)**2
b_MSD = torch.mean(b_diffsq)

diffsq = torch.diff(torch.norm(qT, dim=-1), dim=-1)**2
MSD = torch.mean(diffsq, dim=(1,2))

print('Mean Square Displacement:')
print('Benchmark:', float(b_MSD))

for i in range(MSD.shape[0]):
    print(f'Model {i+1}:', float(MSD[i]))

In [ ]:
def plot_traj_snippet(
    b_vT,                 # [nTrajs, Tb, 3]
    vT,                   # [nModels, nTrajs, Tm, 3]
    var=None,
    L=200,
    dim=0,                # 0=x, 1=y, 2=z
    traj_idx=None,        # choose specific traj or None=random
    start=None,           # choose specific start or None=random
    model_ids=None,       # list of model indices to plot, or None=all
    sharex=True,
):
    """
    Plot a short snippet of one velocity component for a single trajectory:
      - top: benchmark
      - then one subplot per model

    Assumes b_vT and vT are torch tensors or numpy arrays.
    """

    # --- to numpy (works for torch or np) ---
    def to_np(x):
        if hasattr(x, "detach"):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    b = to_np(b_vT)
    m = to_np(vT)

    assert b.ndim == 3 and b.shape[-1] == 3
    assert m.ndim == 4 and m.shape[-1] == 3
    n_models, n_trajs_m, Tm, _ = m.shape
    n_trajs_b, Tb, _ = b.shape
    assert n_trajs_b == n_trajs_m, "Benchmark and model must have same nTrajs."

    if model_ids is None:
        model_ids = list(range(n_models))
    else:
        model_ids = list(model_ids)

    # --- pick trajectory ---
    if traj_idx is None:
        traj_idx = np.random.randint(0, n_trajs_b)

    # --- choose snippet bounds (use min length across bench/models to align) ---
    T_common = min(Tb, Tm)
    if L >= T_common:
        raise ValueError(f"L={L} must be < min(Tb={Tb}, Tm={Tm})={T_common}")

    if start is None:
        start = np.random.randint(0, T_common - L)
    end = start + L

    # --- plot ---
    n_panels = 1 + len(model_ids)
    fig, axes = plt.subplots(n_panels, 1, figsize=(12, 2.2*n_panels), sharex=sharex)

    if n_panels == 1:
        axes = [axes]

    x = np.arange(start, end)

    # benchmark
    axes[0].plot(x, b[traj_idx, start:end, dim], 'black')
    axes[0].set_title(f"Benchmark {var}[{['x','y','z'][dim]}], traj={traj_idx}, snippet=[{start}:{end}]")
    #axes[0].axhline(0.0, linewidth=0.8)

    # models
    for j, mid in enumerate(model_ids, start=1):
        axes[j].plot(x, m[mid, traj_idx, start:end, dim], color=colors[j])
        axes[j].set_title(f"{datasetLabels[mid]} {var}[{['x','y','z'][dim]}], traj={traj_idx}, snippet=[{start}:{end}]")
        #axes[j].axhline(0.0, linewidth=0.8)

    axes[-1].set_xlabel("timestep index")
    for ax in axes:
        if var=='vel':
            ax.set_ylim([-0.5, 0.5])
            ax.set_ylabel("v")
        elif var=='aux':
            ax.set_ylim([-0.15, 0.15])
            ax.set_ylabel("r")


    plt.tight_layout()
    plt.show()

    return traj_idx, start, end

In [ ]:
# random traj, random snippet, v_x
traj_idx, start, _ = plot_traj_snippet(b_vT, vT, var='vel', L=200, dim=0)

In [ ]:
# random traj, random snippet, v_x
plot_traj_snippet(b_rT, rT, var='aux', L=200, dim=0, traj_idx=traj_idx, start=start)